In [1]:
import tequila as tq
from tequila.grouping.compile_groups import compile_commuting_parts
import numpy as np
import openfermion as of
import scipy
from sunrise.measurement import compute_num_meas

mol = tq.Molecule(geometry="h 0.0 0.0 0.0\nh 0.0 0.0 1.5\nh 0.0 0.0 3.0\nh 0.0 0.0 4.5", basis_set="sto-3g").use_native_orbitals() # Linear H4

H = mol.make_hamiltonian()
Hof = H.to_openfermion()
Hsparse = of.linalg.get_sparse_operator(Hof)
v,vv = scipy.sparse.linalg.eigsh(Hsparse, sigma=mol.compute_energy("fci"))
# print(f"error: {mol.compute_energy('fci') - v[0]}") # check
wfn = tq.QubitWaveFunction.from_array(vv[:,0])

measurements = {}

H = mol.make_hamiltonian()
print(f"len(H): {len(H)}")

options = {
    'method': 'si', # lf, rlf, si, ics, lr, fff-lr
    'condition': 'fc', # qwc, fc
    'optimize': 'yes',
    'n_el': mol.n_electrons
}
fc_groups_and_unitaries, sample_ratios = compile_commuting_parts(H, unitary_circuit='improved', options=options)

number_fc_groups = len(fc_groups_and_unitaries)
print(f'method: {options["method"]}, condition: {options["condition"]}')
print('The number of groups to measure is: ', number_fc_groups)
E = tq.ExpectationValue(H=H, U=tq.QCircuit(), optimize_measurements=options)
md = 0
counter = 0
for e in E.get_expectationvalues():
    depth = tq.compile_circuit(e.U).depth
    if depth > md: md = depth

    cnots = 0
    for g in tq.compile_circuit(e.U).gates:
        if g.control:
            cnots += 1
    if cnots > counter: counter = cnots
print("depth overhead: ", md)
print("CNOTs: ", counter)

groups = []
transformations = []
for group, circuit in fc_groups_and_unitaries:
    groups.append(group)
    transformations.append(circuit)
M_tot = compute_num_meas(initial_state=wfn, meas_groups=groups, transformations=transformations)
print(f"M_tot: {M_tot:e}")
measurements[options['method']] = [number_fc_groups, md, counter, M_tot]

/opt/anaconda3/envs/sun/lib/python3.10/site-packages/tequila/quantumchemistry/chemistry_tools.py:367: TequilaWarning: Warning: No units passed with geometry, assuming units are angstrom.
  warnings.warn("Warning: No units passed with geometry, assuming units are angstrom.", TequilaWarning)


len(H): 361
method: si, condition: fc
The number of groups to measure is:  19
depth overhead:  32
CNOTs:  48
M_tot: 3.487122e+04


In [3]:
import tequila as tq
from tequila.grouping.compile_groups import compile_commuting_parts
import numpy as np
import openfermion as of
import scipy
from sunrise.measurement import compute_num_meas

atomic_distances = [0.5, 0.7, 1.0, 1.5, 2.0]
all_results = {}

for d in atomic_distances:
    print(f"\n{'='*50}")
    print(f"Atomic distance: {d} Å")
    print(f"{'='*50}")

    # Scale the 4-atom linear chain by the spacing d
    mol = tq.Molecule(
        geometry=f"h 0.0 0.0 0.0\n"
                 f"h 0.0 0.0 {d}\n"
                 f"h 0.0 0.0 {2*d}\n"
                 f"h 0.0 0.0 {3*d}",
        basis_set="sto-3g"
    ).use_native_orbitals()

    H = mol.make_hamiltonian()
    Hof = H.to_openfermion()
    Hsparse = of.linalg.get_sparse_operator(Hof)
    v, vv = scipy.sparse.linalg.eigsh(Hsparse, sigma=mol.compute_energy("fci"))
    wfn = tq.QubitWaveFunction.from_array(vv[:, 0])

    measurements = {}
    H = mol.make_hamiltonian()
    # print(f"len(H): {len(H)}")

    options = {
        'method': 'si',
        'condition': 'fc',
        'optimize': 'yes',
        'n_el': mol.n_electrons
    }

    fc_groups_and_unitaries, sample_ratios = compile_commuting_parts(
        H, unitary_circuit='improved', options=options
    )
    number_fc_groups = len(fc_groups_and_unitaries)
    print(f'method: {options["method"]}, condition: {options["condition"]}')
    print('The number of groups to measure is: ', number_fc_groups)

    E = tq.ExpectationValue(H=H, U=tq.QCircuit(), optimize_measurements=options)
    md = 0
    counter = 0
    for e in E.get_expectationvalues():
        depth = tq.compile_circuit(e.U).depth
        if depth > md:
            md = depth
        cnots = 0
        for g in tq.compile_circuit(e.U).gates:
            if g.control:
                cnots += 1
        if cnots > counter:
            counter = cnots

    # print("depth overhead: ", md)
    # print("CNOTs: ", counter)

    groups = []
    transformations = []
    for group, circuit in fc_groups_and_unitaries:
        groups.append(group)
        transformations.append(circuit)

    M_tot = compute_num_meas(
        initial_state=wfn, meas_groups=groups, transformations=transformations
    )
    print(f"M_tot: {M_tot:e}")

    measurements[options['method']] = [number_fc_groups, md, counter, M_tot]
    all_results[d] = measurements

# Summary
print("\n\n=== SUMMARY ===")
print(f"{'Dist':>6} | {'Groups':>6} | {'Depth':>5} | {'CNOTs':>5} | {'M_tot':>12}")
print("-" * 45)
for d, meas in all_results.items():
    ng, depth, cnots, mtot = meas['si']
    print(f"{d:>6.1f} | {ng:>6} | {depth:>5} | {cnots:>5} | {mtot:>12.4e}")


Atomic distance: 0.5 Å


/opt/anaconda3/envs/sun/lib/python3.10/site-packages/tequila/quantumchemistry/chemistry_tools.py:367: TequilaWarning: Warning: No units passed with geometry, assuming units are angstrom.
  warnings.warn("Warning: No units passed with geometry, assuming units are angstrom.", TequilaWarning)


method: si, condition: fc
The number of groups to measure is:  20
M_tot: 5.962073e+05

Atomic distance: 0.7 Å
method: si, condition: fc
The number of groups to measure is:  20
M_tot: 1.705854e+05

Atomic distance: 1.0 Å
method: si, condition: fc
The number of groups to measure is:  19
M_tot: 7.844656e+04

Atomic distance: 1.5 Å
method: si, condition: fc
The number of groups to measure is:  19
M_tot: 3.487122e+04

Atomic distance: 2.0 Å
method: si, condition: fc
The number of groups to measure is:  20
M_tot: 1.122170e+04


=== SUMMARY ===
  Dist | Groups | Depth | CNOTs |        M_tot
---------------------------------------------
   0.5 |     20 |    32 |    48 |   5.9621e+05
   0.7 |     20 |    32 |    48 |   1.7059e+05
   1.0 |     19 |    32 |    48 |   7.8447e+04
   1.5 |     19 |    32 |    48 |   3.4871e+04
   2.0 |     20 |    32 |    48 |   1.1222e+04


In [2]:
import tequila as tq
import sunrise as sun
from sunrise.measurement import compute_num_meas
import numpy as np
import openfermion as of
import scipy

atomic_distances = [0.5, 0.7, 1.0, 1.5, 2.0]
all_results = {}

for d in atomic_distances:
    print(f"\n{'='*50}")
    print(f"Atomic distance: {d} Å")
    print(f"{'='*50}")

    # Create the molecule
    mol = tq.Molecule(
        geometry=f"h 0.0 0.0 0.0\n"
                 f"h 0.0 0.0 {d}\n"
                 f"h 0.0 0.0 {2*d}\n"
                 f"h 0.0 0.0 {3*d}",
        basis_set="sto-3g"
    ).use_native_orbitals()

    fci = mol.compute_energy("fci")
    H = mol.make_hamiltonian()

    # Create true wave function
    Hof = H.to_openfermion()
    Hsparse = of.linalg.get_sparse_operator(Hof)
    v, vv = scipy.sparse.linalg.eigsh(Hsparse, sigma=fci)
    wfn = tq.QubitWaveFunction.from_array(vv[:, 0])
    energy = wfn.inner(H * wfn).real

    # Create rotators
    graphs = [
        [(0, 1), (2, 3)],
        [(0, 3), (1, 2)],
        [(0, 2), (1, 3)]
    ]
    rotators = []
    for graph in graphs:
        UR = tq.QCircuit()
        for edge in graph:
            UR += mol.UR(edge[0], edge[1], angle=np.pi/2)
        rotators.append(UR)

    # Apply the measurement protocol
    result = sun.measurement.rotate_and_hcb(
        molecule=mol, rotators=rotators, target=fci,
        initial_state=wfn, silent=True
    )
    print(result)

    # Compute the energy
    energy = 0
    for i, hcb_mol in enumerate(result[0]):
        expval = tq.ExpectationValue(U=rotators[i], H=hcb_mol.make_hamiltonian())
        energy += tq.simulate(expval, initial_state=wfn)
    print(f"Energy of the accumulated HCB contributions: {energy}")
    print(f"Error: {energy - fci}")

    M = 1
    for hcb_mol in result[0]:
        M += compute_num_meas(initial_state=wfn, is_hcb=True, hcb_mol=hcb_mol)
    print(f"M_tot = {M:.3e}")

    all_results[d] = {"fci": fci, "energy": energy, "error": energy - fci, "M_tot": M}

# Summary
print("\n\n=== SUMMARY ===")
print(f"{'Dist':>6} | {'FCI Energy':>12} | {'HCB Energy':>12} | {'Error':>12} | {'M_tot':>10}")
print("-" * 62)
for d, res in all_results.items():
    print(f"{d:>6.1f} | {res['fci']:>12.6f} | {res['energy']:>12.6f} | {res['error']:>12.2e} | {res['M_tot']:>10.3e}")


Atomic distance: 0.5 Å
([<tequila.quantumchemistry.qc_base.QuantumChemistryBase object at 0x15c591ea0>, <tequila.quantumchemistry.qc_base.QuantumChemistryBase object at 0x15c5901c0>, <tequila.quantumchemistry.qc_base.QuantumChemistryBase object at 0x15c591690>], <tequila.quantumchemistry.qc_base.QuantumChemistryBase object at 0x15c593fa0>)
Energy of the accumulated HCB contributions: -1.670750632903616
Error: -0.017633680963524023
M_tot = 1.202e+06

Atomic distance: 0.7 Å
([<tequila.quantumchemistry.qc_base.QuantumChemistryBase object at 0x15c614e80>, <tequila.quantumchemistry.qc_base.QuantumChemistryBase object at 0x15c51c7f0>, <tequila.quantumchemistry.qc_base.QuantumChemistryBase object at 0x15c51d210>], <tequila.quantumchemistry.qc_base.QuantumChemistryBase object at 0x15c617460>)
Energy of the accumulated HCB contributions: -2.111428536663652
Error: -0.004431621568370403
M_tot = 3.340e+05

Atomic distance: 1.0 Å
([<tequila.quantumchemistry.qc_base.QuantumChemistryBase object at 0

```
=== SORTED INSERTION ===
  Dist | Groups | Depth | CNOTs |        M_tot
---------------------------------------------
   0.5 |     20 |    32 |    48 |   5.9621e+05
   0.7 |     20 |    32 |    48 |   1.7059e+05
   1.0 |     19 |    32 |    48 |   7.8447e+04
   1.5 |     19 |    32 |    48 |   3.4871e+04
   2.0 |     20 |    32 |    48 |   1.1222e+04


   === HCB METHOD ===
  Dist |   FCI Energy |   HCB Energy |        Error |      M_tot
--------------------------------------------------------------
   0.5 |    -1.653117 |    -1.670751 |    -1.76e-02 |  1.202e+06
   0.7 |    -2.106997 |    -2.111429 |    -4.43e-03 |  3.340e+05
   1.0 |    -2.166387 |    -2.165232 |     1.16e-03 |  7.561e+04
   1.5 |    -1.996150 |    -1.995129 |     1.02e-03 |  2.282e+04
   2.0 |    -1.897781 |    -1.897600 |     1.81e-04 |  8.772e+03
```

In [ ]:
import tequila as tq
from tequila.grouping.compile_groups import compile_commuting_parts
import numpy as np
import openfermion as of
import scipy
import sunrise as sun
from sunrise.measurement import compute_num_meas
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

atomic_distances = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]
all_results = {}

for d in atomic_distances:
    print(f"\n{'='*50}")
    print(f"Atomic distance: {d} Å")
    print(f"{'='*50}")

    mol = tq.Molecule(
        geometry=f"h 0.0 0.0 0.0\n"
                 f"h 0.0 0.0 {d}\n"
                 f"h 0.0 0.0 {2*d}\n"
                 f"h 0.0 0.0 {3*d}",
        basis_set="sto-3g"
    ).use_native_orbitals()

    fci = mol.compute_energy("fci")
    H = mol.make_hamiltonian()

    Hof = H.to_openfermion()
    Hsparse = of.linalg.get_sparse_operator(Hof)
    v, vv = scipy.sparse.linalg.eigsh(Hsparse, sigma=fci)
    wfn = tq.QubitWaveFunction.from_array(vv[:, 0])

    # ── Helper: run a commuting-groups method ──────────────────────────────────
    def run_commuting_method(method_key, condition):
        opts = {
            'method': method_key,
            'condition': condition,
            'optimize': 'yes',
            'n_el': mol.n_electrons
        }
        groups_and_unitaries, _ = compile_commuting_parts(
            H, unitary_circuit='improved', options=opts
        )
        groups, transformations = [], []
        for group, circuit in groups_and_unitaries:
            groups.append(group)
            transformations.append(circuit)
        M = compute_num_meas(
            initial_state=wfn, meas_groups=groups, transformations=transformations
        )
        n_groups = len(groups)
        print(f"[{method_key.upper()}/{condition}]  groups: {n_groups}  M_tot: {M:e}")
        return M, n_groups

    # ── Method 1: SI (fc) ──────────────────────────────────────────────────────
    M_tot_si, _ = run_commuting_method('si', 'fc')

    # ── Method 2: LF (fc) ──────────────────────────────────────────────────────
    M_tot_lf, _ = run_commuting_method('lf', 'fc')

    # ── Method 3: HCB (rotate_and_hcb) ────────────────────────────────────────
    graphs = [
        [(0, 1), (2, 3)],
        [(0, 3), (1, 2)],
        [(0, 2), (1, 3)]
    ]
    rotators = []
    for graph in graphs:
        UR = tq.QCircuit()
        for edge in graph:
            UR += mol.UR(edge[0], edge[1], angle=np.pi/2)
        rotators.append(UR)

    result = sun.measurement.rotate_and_hcb(
        molecule=mol, rotators=rotators, target=fci,
        initial_state=wfn, silent=True
    )

    M_tot_hcb = 1
    for hcb_mol in result[0]:
        M_tot_hcb += compute_num_meas(initial_state=wfn, is_hcb=True, hcb_mol=hcb_mol)
    print(f"[HCB]             M_tot: {M_tot_hcb:.3e}")

    all_results[d] = {
        "M_tot_si":  M_tot_si,
        "M_tot_lf":  M_tot_lf,
        "M_tot_hcb": M_tot_hcb,
    }

# ── Summary table ──────────────────────────────────────────────────────────────
print("\n\n=== SUMMARY ===")
print(f"{'Dist':>6} | {'M_tot (SI)':>14} | {'M_tot (LF)':>14} | {'M_tot (HCB)':>14}")
print("-" * 58)
for d, res in all_results.items():
    print(f"{d:>6.1f} | {res['M_tot_si']:>14.4e} | {res['M_tot_lf']:>14.4e} | {res['M_tot_hcb']:>14.4e}")

# ── Plot ───────────────────────────────────────────────────────────────────────
distances = list(all_results.keys())
M_si  = [all_results[d]["M_tot_si"]  for d in distances]
M_lf  = [all_results[d]["M_tot_lf"]  for d in distances]
M_hcb = [all_results[d]["M_tot_hcb"] for d in distances]

fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor("#0d0d0f")
ax.set_facecolor("#13131a")

methods = [("SI", M_si, "#e05c5c"), ("LF", M_lf, "#5c9de0"), ("HCB", M_hcb, "#5ce09d")]
for label, data, color in methods:
    ax.plot(distances, data, marker="o", linewidth=2.2, markersize=7,
            color=color, label=label, zorder=3)

ax.set_yscale("log")
ax.set_xlabel("Atomic distance (Å)", color="#cccccc", fontsize=12, labelpad=8)
ax.set_ylabel(r"$M_{\mathrm{tot}}$", color="#cccccc", fontsize=13, labelpad=8)
ax.set_title("Measurement cost comparison\nH₄ linear chain (STO-3G)",
             color="#eeeeee", fontsize=13, pad=12)

ax.tick_params(colors="#aaaaaa", labelsize=10)
for spine in ax.spines.values():
    spine.set_edgecolor("#333344")
ax.grid(True, which="both", linestyle="--", linewidth=0.5, color="#2a2a3a", zorder=0)
ax.legend(frameon=True, facecolor="#1e1e2e", edgecolor="#444466",
          labelcolor="#dddddd", fontsize=11, loc="best")

plt.tight_layout()
plot_path = "M_tot_comparison.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"\nPlot saved to {plot_path}")


Atomic distance: 0.25 Å
[SI/fc]  groups: 19  M_tot: 4.116322e+06
[LF/fc]  groups: 28  M_tot: 1.208227e+07
[HCB]             M_tot: 7.423e+06

Atomic distance: 0.5 Å
[SI/fc]  groups: 20  M_tot: 5.962073e+05
[LF/fc]  groups: 28  M_tot: 1.812671e+06
[HCB]             M_tot: 1.202e+06

Atomic distance: 0.75 Å
[SI/fc]  groups: 20  M_tot: 1.444632e+05
[LF/fc]  groups: 28  M_tot: 4.967597e+05
[HCB]             M_tot: 2.537e+05

Atomic distance: 1.0 Å
[SI/fc]  groups: 19  M_tot: 7.844656e+04
[LF/fc]  groups: 28  M_tot: 2.540020e+05
[HCB]             M_tot: 7.561e+04

Atomic distance: 1.25 Å
[SI/fc]  groups: 19  M_tot: 5.170171e+04
[LF/fc]  groups: 28  M_tot: 1.692849e+05
[HCB]             M_tot: 3.797e+04

Atomic distance: 1.5 Å
[SI/fc]  groups: 19  M_tot: 3.487122e+04
[LF/fc]  groups: 28  M_tot: 1.189497e+05
[HCB]             M_tot: 2.282e+04

Atomic distance: 1.75 Å
[SI/fc]  groups: 20  M_tot: 2.127201e+04
[LF/fc]  groups: 28  M_tot: 7.761868e+04
[HCB]             M_tot: 1.343e+04

Atomic d